In [ ]:
#@title { display-mode: "form" }
from IPython.display import display, HTML

display(HTML("""
<style>
  .kh-header {
    display: grid;
    grid-template-columns: 1fr auto;
    grid-template-areas: "title img" "body img";
    column-gap: 20px;
    align-items: center;
  }
  .kh-title { grid-area: title; color: #1a73e8; font-size: 2.6em; font-weight: 700; margin: 0 0 8px; }
  .kh-body  { grid-area: body; line-height: 1.6; }
  .kh-body ul { margin: 6px 0 0 1.2em; padding: 0; }
  .kh-img   { grid-area: img; width: 3.75cm; height: 3.75cm; object-fit: cover; border-radius: 12px; margin-right: 8vw; }

  @media (max-width: 600px) {
    .kh-header {
      grid-template-columns: 1fr auto;
      grid-template-areas: "title img" "body body";
      column-gap: 12px;
      align-items: center;
    }
    .kh-title {
      font-size: 1.8em;
      margin: 0;
    }
    .kh-img {
      width: 2.2cm;
      height: 2.2cm;
      margin: 0;
      position: relative;
      left: -30%;
      right: 15%;
    }
  }
</style>

<div class="kh-header">
  <h1 class="kh-title">Al-Khwarizmi</h1>
  <img class="kh-img" alt="Al-Khwarizmi banner"
       src="https://huggingface.co/mzoelfakar/Al-Khwarizmi-3B/resolve/main/banner.png">
  <div class="kh-body">
    Hello,<br>
    Before chatting:
    <ul>
      <li><b>Sign in</b> to your Google account</li>
      <li>Runtime &gt; <b>Change runtime type &gt; T4 GPU</b> (good performance)</li>
    </ul>
  </div>
</div>
"""))

In [ ]:
# @title **↓↓** Click Play *(please wait ~2 mins)*
import warnings
warnings.filterwarnings("ignore")
import logging
logging.disable(logging.WARNING)

import transformers.utils.logging
transformers.utils.logging.set_verbosity_error()
import os, time
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import re, ast, operator, unicodedata
from fractions import Fraction
from transformers import StoppingCriteria, StoppingCriteriaList

import sys
from io import StringIO

class HiddenPrints:
    def __enter__(self):
        self._original_stdout = sys.stdout
        sys.stdout = StringIO()

    def __exit__(self, exc_type, exc_val, exc_tb):
        sys.stdout = self._original_stdout

from google.colab import output


def clean_text(text):
    text = re.sub(r'<<.*?>>', '', text)
    text = re.sub(
        r'#### (.*)',
        r'<div class="final-answer" style="display: block; margin-top: 10px;">\1</div>',
        text
    )
    # Socratic separator " ** " -> line break
    text = re.sub(r'\s+\*\*[ \t]*|(?<=\?)\*\*[ \t]*', '\n', text)
    text = re.sub(r'(?<=[\w])(\s*)\*(\s*)(?=[\w])', r'\1\\*\2', text)
    return text


def detect_direction(text):
    """
    Determine base text direction (rtl/ltr) for BiDi rendering of message
    content only. Counts strong-directional characters: Arabic/Hebrew Unicode
    blocks vs Latin letters, ignoring neutral characters (digits, punctuation,
    whitespace). Falls back to 'ltr' when text is empty or has no strong
    directional characters yet (e.g. a message still streaming in).
    """
    if not text:
        return "ltr"

    stripped = re.sub(r'<[^>]+>', '', text)

    rtl_chars = re.findall(
        r'[\u0591-\u07FF\uFB1D-\uFDFD\uFE70-\uFEFC]',
        stripped
    )
    ltr_chars = re.findall(r'[A-Za-z]', stripped)

    if len(rtl_chars) > len(ltr_chars):
        return "rtl"

    return "ltr"


# ── Calculator-in-the-loop decoding ──────────────────────────────────────
USE_CALCULATOR = True
MAX_NEW_TOKENS = 500
OPEN_EQ = re.compile(r'<<([^<>=]*)=([^<>]*)$')

_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv
}


def _ev(n):
    """Exact arithmetic on a parsed expression: numbers, + - * /, ** (small integer powers), parentheses, unary +/-."""
    if isinstance(n, ast.Expression):
        return _ev(n.body)

    if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)) and not isinstance(n.value, bool):
        return Fraction(str(n.value))

    if isinstance(n, ast.BinOp) and type(n.op) in _OPS:
        return _OPS[type(n.op)](_ev(n.left), _ev(n.right))

    if isinstance(n, ast.BinOp) and isinstance(n.op, ast.Pow):
        b, e = _ev(n.left), _ev(n.right)
        if e.denominator != 1 or abs(e) > 64:
            raise ValueError("unsupported power")
        return b ** int(e)

    if isinstance(n, ast.UnaryOp) and isinstance(n.op, (ast.UAdd, ast.USub)):
        v = _ev(n.operand)
        return -v if isinstance(n.op, ast.USub) else v

    raise ValueError("unsupported expression")


def calc(expr):
    """Safely evaluate an expression written by the model; returns the result as text, or None if it can't."""
    expr = re.sub(
        r'\d',
        lambda m: str(unicodedata.digit(m.group())),
        expr
    )

    expr = (
        expr.replace("×", "*")
        .replace("÷", "/")
        .replace("$", "")
        .replace(",", "")
        .replace(" ", "")
    )

    if not expr:
        return None

    try:
        v = _ev(ast.parse(expr, mode="eval"))
    except Exception:
        return None

    return (
        str(int(v))
        if v.denominator == 1
        else f"{float(v):.6f}".rstrip("0").rstrip(".")
    )


def splice_calc(seg):
    """If seg ends inside an open <<expr= block: cut it right after '=' and append the exact result."""
    m = OPEN_EQ.search(seg)
    val = calc(m.group(1)) if m else None
    return (seg[:m.start(2)] + val, True) if val is not None else (seg, False)


class CalcStop(StoppingCriteria):
    """Stops generation as soon as the model has written '=' inside a <<...>> block."""
    def __init__(self, start_len):
        self.start_len, self.hit = start_len, False

    def __call__(self, input_ids, scores, **kwargs):
        if USE_CALCULATOR and not self.hit:
            tail = tokenizer.decode(
                input_ids[0, max(self.start_len, input_ids.shape[1] - 48):],
                skip_special_tokens=True
            )
            self.hit = bool(OPEN_EQ.search(tail))

        return torch.full(
            (input_ids.shape[0],),
            self.hit,
            dtype=torch.bool,
            device=input_ids.device
        )


def stream_reply(prompt, on_text):
    """Generate a reply; whenever the model writes '=' inside <<expr=, pause, insert the exact result, resume."""
    response, budget = "", MAX_NEW_TOKENS

    while budget > 0:
        inputs = tokenizer(
            prompt + response,
            return_tensors="pt"
        ).to(model.device)

        n_in = inputs["input_ids"].shape[1]
        stop = CalcStop(n_in)

        streamer = TextIteratorStreamer(
            tokenizer,
            skip_prompt=True,
            skip_special_tokens=True
        )

        box = {}

        def work():
            box["ids"] = model.generate(
                **inputs,
                streamer=streamer,
                max_new_tokens=budget,
                temperature=0.7,
                do_sample=True,
                stopping_criteria=StoppingCriteriaList([stop])
            )

        thread = Thread(target=work)
        thread.start()

        seg = ""

        for piece in streamer:
            seg += piece
            on_text(response + seg)

        thread.join()

        new_ids = box["ids"][0, n_in:]
        budget -= len(new_ids)

        seg = tokenizer.decode(
            new_ids,
            skip_special_tokens=True
        )

        text, _ = splice_calc(seg) if stop.hit else (seg, False)
        response += text

        if not stop.hit or len(new_ids) == 0:
            break

    return response


# ── Flicker-free chat rendering ──────────────────────────────────────────
CHAT_STYLE = """
<style>
    .chat-container {
        line-height: 1.5;
    }

    .ai-msg, .user-msg {
        font-size: 1.2em !important;
        font-weight: bold !important;
        margin-bottom: 15px;
        display: flex !important;
        align-items: flex-start;
        direction: ltr !important;
        text-align: left !important;
        width: 100%;
        box-sizing: border-box;
    }

    .ai-msg {
        color: #1a73e8 !important;
    }

    .user-msg {
        color: #35A630 !important;
    }

    .msg-label {
        direction: ltr !important;
        unicode-bidi: isolate;
        flex: 0 0 auto;
        white-space: nowrap;
        display: inline !important;
    }

    .msg-content {
        flex: 1 1 auto;
        min-width: 0;
        unicode-bidi: plaintext;
        display: block !important;
        box-sizing: border-box;
    }

    .final-answer {
        color: #1a73e8 !important;
        font-weight: 900 !important;
        font-size: 1.1em;
    }

    .ai-msg p, .user-msg p,
    .ai-msg div, .user-msg div {
        font-size: 1em !important;
        font-weight: inherit !important;
        color: inherit !important;
    }

    .final-answer {
        display: block !important;
    }
</style>
"""


GREETING_HTML = (
    '<div class="ai-msg">'
    '<span class="msg-label">Al-Khwarizmi:</span>'
    '<span class="msg-content" '
    'style="direction:ltr; text-align:left;">'
    'Hi, I am Al-Khwarizmi, how can I help you?'
    '</span>'
    '</div>'
)

THINKING_HTML = (
    '<div class="ai-msg">'
    '<span class="msg-label">Al-Khwarizmi:</span>'
    '<span class="msg-content" '
    'style="direction:ltr; text-align:left;">'
    '...'
    '</span>'
    '</div>'
)


def user_block(u):
    direction = detect_direction(u)

    if direction == "rtl":
        content_style = (
            "direction:rtl;"
            "text-align:right;"
            "padding-right:1cm;"
        )
    else:
        content_style = (
            "direction:ltr;"
            "text-align:left;"
        )

    return (
        '<div class="user-msg">'
        '<span class="msg-label">You:</span>'
        f'<span class="msg-content" style="{content_style}">{u}</span>'
        '</div>'
    )


def ai_block(a, streaming=False):
    if streaming:
        # Hide half-written bits while streaming
        a = re.sub(
            r'<<(?:(?!>>).)*$|<$|#{1,4}\s*$|[ \t]+\*{1,2}$',
            '',
            a,
            flags=re.S
        )

    a = clean_text(a)

    rendered = markdown.markdown(
        a,
        extensions=["fenced_code", "tables", "nl2br"]
    ).replace("<p>", "").replace("</p>", "")

    direction = detect_direction(a)

    if direction == "rtl":
        content_style = (
            "direction:rtl;"
            "text-align:right;"
            "padding-right:1cm;"
        )
    else:
        content_style = (
            "direction:ltr;"
            "text-align:left;"
        )

    return (
        '<div class="ai-msg">'
        '<span class="msg-label">Al-Khwarizmi:</span>'
        f'<span class="msg-content" style="{content_style}">'
        f'{rendered}'
        '</span>'
        '</div>'
    )


def transcript_html():
    return (
        CHAT_STYLE
        + GREETING_HTML
        + "".join(
            user_block(u) + ai_block(a)
            for u, a in history
        )
    )


def run_chat_loop():
    output.clear(wait=True)
    display(HTML(transcript_html()))

    while True:
        print("\n")
        user_input = input("You: ")

        if user_input.lower() == "exit":
            break

        recent_history = history[-10:]

        messages = [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            }
        ]

        for u, a in recent_history:
            messages.append({
                "role": "user",
                "content": u
            })
            messages.append({
                "role": "assistant",
                "content": a
            })

        messages.append({
            "role": "user",
            "content": user_input
        })

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        # Everything above the reply is static for this turn
        prefix = transcript_html() + user_block(user_input)
        chat_id = str(uuid.uuid4())

        # One atomic swap
        output.clear(wait=True)
        display(
            HTML(prefix + THINKING_HTML),
            display_id=chat_id
        )

        last_paint = [0.0]

        def show(text, final=False):
            now = time.time()

            if not final and now - last_paint[0] < 0.08:
                return

            last_paint[0] = now

            update_display(
                HTML(
                    prefix
                    + ai_block(
                        text,
                        streaming=not final
                    )
                ),
                display_id=chat_id
            )

        response = stream_reply(prompt, show)
        show(response, final=True)

        history.append((user_input, response))


try:
    !pip install transformers torch markdown -q

    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        TextIteratorStreamer
    )

    from IPython.display import (
        display,
        HTML,
        update_display
    )

    import torch
    from threading import Thread
    from datetime import date
    import html, uuid, re
    import markdown

    # Hide every download bar except the "Fetching N files" one
    from huggingface_hub.utils import disable_progress_bars

    for g in [
        "huggingface_hub.http_get",
        "huggingface_hub.xet_get",
        "huggingface_hub.snapshot_download.transfer",
        "huggingface_hub.snapshot_download"
    ]:
        disable_progress_bars(g)

    # Rename "Fetching N files" -> "Loading"
    import huggingface_hub._snapshot_download as _sd

    class LoadingBar(_sd.hf_tqdm):
        def __init__(self, *args, **kwargs):
            if str(kwargs.get("desc", "")).startswith("Fetching"):
                kwargs["desc"] = "Loading"
                kwargs["bar_format"] = "{desc}: {percentage:3.0f}%|{bar}|"
            super().__init__(*args, **kwargs)

    _sd.hf_tqdm = LoadingBar

    # Hide transformers' own "Loading weights" bar
    transformers.utils.logging._tqdm_active = False

    model_name = "mzoelfakar/Al-Khwarizmi-3B"

    with HiddenPrints():
        transformers.utils.logging.set_verbosity_error()

        tokenizer = AutoTokenizer.from_pretrained(model_name)

        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            dtype=torch.bfloat16,
            device_map="auto"
        )

    output.clear()

    SYSTEM_PROMPT = f"""You are a professional math tutor called Al-Khwarizmi.

You are always careful. Before answering:

- Use only the numbers and quantities explicitly stated in the problem. Do not introduce, assume, or carry over any value that wasn't given.
- Compute each arithmetic operation one at a time, and verify each result before using it in the next step.
- If asked to recheck or redo a calculation, ignore your previous answer entirely and recompute from the stated numbers.
- Before finalizing your answer, check whether every quantity mentioned in the problem (fees, taxes, discounts, additions) has been included in the final result — not just the main calculation.
- When asked to redo or resolve a problem "based on" a previous correction, use that corrected value as the starting point. Do not revert to an earlier, uncorrected path.
- Avoid sharing numbers only, always include text explanations. Apologize if you don't know the answer.

LANGUAGE RULES: You MUST ALWAYS reply in the same language and script from the user's last message, NEVER use another lanaguage, examples:

Query: مرحبا
Reply: اهلا كيف يمكنني مساعدك؟

Query: اسمك إيه؟
Reply: الخوارزمي

Query: Bojour
Reply: Bonjour ! Comment ça va ?

REFRAIN from repeating unnecessary information.
NEVER help with any topics other than math like the weather, cooking, sports, etc. If asked, redirect gently.

Today's date is {date.today().strftime('%B %d, %Y')}."""

    history = []

    run_chat_loop()

except BaseException:
    pass

finally:
    time.sleep(0.3)
    output.clear()

*An AI math tutor named after Muhammad al-Khwarizmi, the 9th-century mathematician whose name is the origin of the word "algorithm".*

*Note: this model is trained on simple math problems and may make mistakes on complex ones. Model card [available here](https://huggingface.co/mzoelfakar/Al-Khwarizmi-3B).*

*By [Mohamed Zoelfakar](https://www.linkedin.com/in/mzoelfakar/)*